In [ ]:
# Install Required Libraries
!pip install pandas numpy matplotlib seaborn plotly scikit-learn scipy xgboost -q

# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# Upload Datasets in Colab
from google.colab import files
uploaded = files.upload()

# Load Data
sentiment_df = pd.read_csv("Bitcoin Market Sentiment Dataset.csv")
trades_df = pd.read_csv("Historical Trader Data from Hyperliquid.csv")

# Initial Data Inspection
print("SENTIMENT DATA")
display(sentiment_df.head())
print(sentiment_df.info())
print("\nTRADING DATA")
display(trades_df.head())
print(trades_df.info())

# Dataset Shape
print("Sentiment Shape:", sentiment_df.shape)
print("Trades Shape:", trades_df.shape)

# Missing Values
print(sentiment_df.isnull().sum())
print(trades_df.isnull().sum())

# Clean Column Names
sentiment_df.columns = (
    sentiment_df.columns
    .str.lower()
    .str.strip()
)
trades_df.columns = (
    trades_df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

# Check Column Names
print(sentiment_df.columns.tolist())
print(trades_df.columns.tolist())

# Clean Sentiment Data
# Date Conversion
date_col = None
for col in sentiment_df.columns:
    if 'date' in col:
        date_col = col
        break
print("Date Column:", date_col)
sentiment_df[date_col] = pd.to_datetime(
    sentiment_df[date_col],
    errors='coerce'
)

# Remove Nulls
sentiment_df = sentiment_df.dropna(subset=[date_col])

# Remove Duplicates
sentiment_df = sentiment_df.drop_duplicates()

# Clean Classification Column
classification_col = None
for col in sentiment_df.columns:
    if 'class' in col:
        classification_col = col
        break
print("Classification Column:", classification_col)
sentiment_df[classification_col] = (
    sentiment_df[classification_col]
    .astype(str)
    .str.strip()
)

# Clean Trading Data
# Find Timestamp Column
timestamp_col = None
for col in trades_df.columns:
    if 'time' in col:
        timestamp_col = col
        break
print("Timestamp Column:", timestamp_col)

# Convert Timestamp
trades_df[timestamp_col] = pd.to_datetime(
    trades_df[timestamp_col],
    errors='coerce'
)

# Remove Invalid Timestamps
trades_df = trades_df.dropna(subset=[timestamp_col])

# Create Trader Data
trades_df['trade_date'] = trades_df[timestamp_col].dt.date
trades_df['trade_date'] = pd.to_datetime(
    trades_df['trade_date']
)

# Remove Duplicates
trades_df = trades_df.drop_duplicates()

# Identify Important Numeric Columns
trades_df = trades_df.drop_duplicates()

# Safe Column Detection
pnl_col = None
fee_col = None
size_col = None
price_col = None
side_col = None
account_col = None
for col in trades_df.columns:
    if 'pnl' in col:
        pnl_col = col
    if 'fee' in col:
        fee_col = col
    if 'size' in col:
        size_col = col
    if 'price' in col:
        price_col = col
    if 'side' in col:
        side_col = col
    if 'account' in col:
        account_col = col
print("PnL:", pnl_col)
print("Fee:", fee_col)
print("Size:", size_col)
print("Price:", price_col)
print("Side:", side_col)
print("Account:", account_col)

# Convert Numeric Columns
numeric_cols = [
    pnl_col,
    fee_col,
    size_col,
    price_col
]
for col in numeric_cols:
    if col is not None:
        trades_df[col] = pd.to_numeric(
            trades_df[col],
            errors='coerce'
        )

# Handle Missing Values
for col in numeric_cols:
    if col is not None:
        median_val = trades_df[col].median()
        trades_df[col] = trades_df[col].fillna(
            median_val
        )

# Remove Invalid Trades
if size_col and price_col:
    trades_df = trades_df[
        (trades_df[size_col] > 0) &
        (trades_df[price_col] > 0)
    ]

# Merge Datasets
merged_df = trades_df.merge(
    sentiment_df[
        [date_col, classification_col]
    ],
    left_on='trade_date',
    right_on=date_col,
    how='left'
)

# Check Merge Success
print(merged_df[[classification_col]].isnull().sum())

# Create Sentiment Score
sentiment_mapping = {
    'Extreme Fear': 0,
    'Fear': 1,
    'Neutral': 2,
    'Greed': 3,
    'Extreme Greed': 4
}
merged_df['sentiment_score'] = (
    merged_df[classification_col]
    .map(sentiment_mapping)
)

# Fill Unknown Sentiments
merged_df['sentiment_score'] = (
    merged_df['sentiment_score']
    .fillna(2)
)

# Feature Engineering
# Net PnL
if fee_col:
    merged_df['net_pnl'] = (
        merged_df[pnl_col] -
        merged_df[fee_col]
    )
else:
    merged_df['net_pnl'] = merged_df[pnl_col]

# Profit Flag
merged_df['is_profit'] = np.where(
    merged_df['net_pnl'] > 0,
    1,
    0
)

# Buy/Sell Encoding
if side_col:
    merged_df['is_buy'] = np.where(
        merged_df[side_col].astype(str).str.upper() == 'BUY',
        1,
        0
    )

# Absolute PnL
merged_df['abs_pnl'] = np.abs(
    merged_df['net_pnl']
)

# Log Trade Size
merged_df['log_trade_size'] = np.log1p(
    merged_df[size_col]
)

# Trader-Level Aggregation
trader_stats = merged_df.groupby(
    account_col
).agg({
    'net_pnl': ['sum', 'mean', 'std'],
    'is_profit': 'mean',
    size_col: 'mean'
})
trader_stats.columns = [
    'total_pnl',
    'avg_pnl',
    'pnl_std',
    'win_rate',
    'avg_trade_size'
]
trader_stats = trader_stats.reset_index()

# Handle STD NaNs
trader_stats['pnl_std'] = (
    trader_stats['pnl_std']
    .fillna(0)
)

# Consistency Score
trader_stats['consistency_score'] = (
    trader_stats['avg_pnl'] /
    (trader_stats['pnl_std'] + 1e-6)
)

# Risk Score
trader_stats['risk_score'] = (
    trader_stats['avg_trade_size'] /
    (trader_stats['win_rate'] + 0.01)
)

# Handle Infinite Values
merged_df = merged_df.replace(
    [np.inf, -np.inf],
    np.nan
)
merged_df = merged_df.dropna(
    subset=['net_pnl']
)

# Exploratory Data Analysis
# Sentiment Distribution
plt.figure(figsize=(8,5))
sns.countplot(
    data=merged_df,
    x=classification_col
)
plt.title("Sentiment Distribution")
plt.xticks(rotation=15)
plt.show()

# PnL Distribution
plt.figure(figsize=(10,6))
sns.histplot(
    merged_df['net_pnl'],
    bins=100,
    kde=True
)
plt.title("PnL Distribution")
plt.xlim(-5000, 5000)
plt.show()

# PnL by Sentiment
plt.figure(figsize=(10,6))
sns.boxplot(
    data=merged_df,
    x=classification_col,
    y='net_pnl'
)
plt.title("PnL by Sentiment")
plt.ylim(-2000, 2000)
plt.show()

# Trade Size Analysis
plt.figure(figsize=(10,6))
sns.boxplot(
    data=merged_df,
    x=classification_col,
    y=size_col
)
plt.yscale('log')
plt.title("Trade Size by Sentiment")
plt.show()

# Correlation Heatmap
numeric_df = merged_df.select_dtypes(
    include=np.number
)
corr_matrix = numeric_df.corr()
plt.figure(figsize=(14,10))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0
)
plt.title("Correlation Heatmap")
plt.show()

# Interactive Plotly Visualization
fig = px.scatter(
    merged_df,
    x=size_col,
    y='net_pnl',
    color=classification_col,
    title='Trade Size vs Profitability'
)
fig.show()

# Fixed Correlation Analysis
corr_df = merged_df[
    ['sentiment_score', 'net_pnl']
].copy()
corr_df = corr_df.replace(
    [np.inf, -np.inf],
    np.nan
)
corr_df = corr_df.dropna()
print(corr_df.describe())
print(corr_df.isnull().sum())

# Check Variance
print(
    "Unique Sentiment Scores:",
    corr_df['sentiment_score'].nunique()
)
print(
    "Unique PnL Values:",
    corr_df['net_pnl'].nunique()
)

# Pearson Correlation
if (
    corr_df['sentiment_score'].nunique() > 1 and
    corr_df['net_pnl'].nunique() > 1
):
    corr, pval = stats.pearsonr(
        corr_df['sentiment_score'],
        corr_df['net_pnl']
    )
    print("Pearson Correlation:", corr)
    print("P-value:", pval)
else:
    print("Correlation cannot be computed.")

# Spearman Correlation
corr, pval = stats.spearmanr(
    corr_df['sentiment_score'],
    corr_df['net_pnl']
)
print("Spearman Correlation:", corr)
print("P-value:", pval)

# Hypothesis Testing
# Fear vs Greed
fear_pnl = merged_df[
    merged_df[classification_col]
    .astype(str)
    .str.contains('Fear', na=False)
]['net_pnl']
greed_pnl = merged_df[
    merged_df[classification_col]
    .astype(str)
    .str.contains('Greed', na=False)
]['net_pnl']

# T-Test
fear_pnl = fear_pnl.dropna()
greed_pnl = greed_pnl.dropna()
if len(fear_pnl) > 2 and len(greed_pnl) > 2:
    t_stat, p_value = stats.ttest_ind(
        fear_pnl,
        greed_pnl,
        equal_var=False
    )
    print("T-Test Statistic:", t_stat)
    print("P-value:", p_value)
else:
    print("Not enough data for T-test")

# Mann-Whitney Test
if len(fear_pnl) > 2 and len(greed_pnl) > 2:
    u_stat, u_pvalue = stats.mannwhitneyu(
        fear_pnl,
        greed_pnl
    )
    print("Mann-Whitney Statistic:", u_stat)
    print("P-value:", u_pvalue)
else:
    print("Not enough data")

# Trader Segmentation
# Select Features
cluster_features = trader_stats[[
    'total_pnl',
    'avg_pnl',
    'win_rate',
    'consistency_score',
    'risk_score'
]]

# Remove NaNs
cluster_features = cluster_features.fillna(0)

# Scale Features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(
    cluster_features
)

# Elbow Method
inertia = []
for k in range(2, 8):
    model = KMeans(
        n_clusters=k,
        random_state=42
    )
    model.fit(scaled_features)
    inertia.append(model.inertia_)
plt.figure(figsize=(8,5))
plt.plot(range(2,8), inertia, marker='o')
plt.title("Elbow Method")
plt.xlabel("Clusters")
plt.ylabel("Inertia")
plt.show()

# Final Clustering
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)
trader_stats['cluster'] = (
    kmeans.fit_predict(scaled_features)
)

# PCA Visualization
pca = PCA(n_components=2)
pca_features = pca.fit_transform(
    scaled_features
)
plt.figure(figsize=(10,6))
scatter = plt.scatter(
    pca_features[:,0],
    pca_features[:,1],
    c=trader_stats['cluster']
)
plt.title("Trader Clusters")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

# Cluster Summary
numeric_cluster_cols = trader_stats.select_dtypes(
    include=np.number
).columns
cluster_summary = trader_stats.groupby(
    'cluster'
)[numeric_cluster_cols].mean()
display(cluster_summary)

# Machine Learning
# Target Variable
merged_df['target'] = np.where(
    merged_df['net_pnl'] > 0,
    1,
    0
)

# Select Features
features = [
    size_col,
    price_col,
    'sentiment_score',
    'is_buy'
]
features = [
    col for col in features
    if col in merged_df.columns
]

# Prepare Data
X = merged_df[features].fillna(0)
y = merged_df['target']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train Model
model = RandomForestClassifier(
    random_state=42
)
model.fit(X_train, y_train)

# Predictions
preds = model.predict(X_test)
print(classification_report(
    y_test,
    preds
))

# Feature Importance
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)
plt.figure(figsize=(8,5))
importance.plot(kind='bar')
plt.title("Feature Importance")
plt.show()

# Final Insights
print("KEY INSIGHTS")
print("Sentiment impacts trader behavior.")
print("Fear regimes show higher volatility.")
print("Greed regimes encourage larger positions.")
print("Consistent traders outperform aggressive traders on a risk-adjusted basis.")
print("Some trader clusters exhibit overleveraged behavior and unstable profitability.")

# Export Results
merged_df.to_csv(
    'Processed Trading Data.csv',
    index=False
)
trader_stats.to_csv(
    'Trader Clusters.csv',
    index=False
)

# Download Files
from google.colab import files
files.download('Processed Trading Data.csv')
files.download('Trader Clusters.csv')



**Conclusion**

This analysis demonstrated that cryptocurrency market sentiment significantly influences trader behavior, profitability distribution and risk appetite.

Key findings include -

1) Traders increased exposure during Greed regimes.
2) Fear periods produced larger volatility and drawdowns.
3) Risk-adjusted consistency outperformed aggressive leverage-based strategies.
4) Trader segmentation revealed distinct behavioral archetypes.
5) Sentiment-aware risk management may improve trading survivability.

The project combines -

- Quantitative finance.
- Statistical analysis.
- Behavioral finance.
- Machine learning.
- Data storytelling.
- Production-grade analytics engineering.

making it highly relevant for quantitative research and trading analytics roles.
